# Notebook 03_01 — Feature Engineering: PCA

Unsupervised linear feature extraction. Fit on training data only, applied to val/test.

**Results saved incrementally to** `results/fe_pca_raw.csv` — resumable on crash.

In [1]:
import os, sys, subprocess

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

CODE_DIR = '/content/UAV' if IN_COLAB else os.environ.get('UAV_CODE_DIR', 'D:/UAV_')

if IN_COLAB and not os.path.isdir(os.path.join(CODE_DIR, '.git')):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/haribawngg/UAV.git', CODE_DIR], check=True)

sys.path.insert(0, os.path.join(CODE_DIR, 'notebook'))
from common import *
from sklearn.decomposition import PCA

print('Imports OK')

CPU: 16 logical cores detected -> N_JOBS=-1 (RF/XGB/LGBM/KNN use all cores)
Imports OK


In [2]:
d = load_data()
X_train, X_val, X_test = d['X_train'], d['X_val'], d['X_test']

METHOD_NAME = 'PCA'
RESULTS_CSV = f'{RESULTS_DIR}fe_pca_raw.csv'

In [3]:
def transform_fn(K):
    pca = PCA(n_components=K, random_state=42)
    Xtr = pca.fit_transform(X_train)   # fit ONLY on train
    Xva = pca.transform(X_val)
    Xte = pca.transform(X_test)
    print(f'  [PCA K={K}] explained variance ratio sum: {pca.explained_variance_ratio_.sum():.4f}')
    return Xtr, Xva, Xte, K

In [4]:
run_experiment_grid(
    method_name=METHOD_NAME,
    transform_fn=transform_fn,
    d=d,
    results_csv_path=RESULTS_CSV
)

  [PCA K=4] explained variance ratio sum: 0.4050
[PCA] K=4 seed=42 DT     F1=0.8897  train=0.2s  inf=0.0001ms/sample  (1/90)
[PCA] K=4 seed=42 RF     F1=0.8903  train=2.1s  inf=0.0028ms/sample  (2/90)
[PCA] K=4 seed=42 XGB    F1=0.8918  train=1.9s  inf=0.0011ms/sample  (3/90)
[PCA] K=4 seed=42 LGBM   F1=0.8893  train=1.0s  inf=0.0030ms/sample  (4/90)
[PCA] K=4 seed=42 KNN    F1=0.7553  train=0.1s  inf=0.0168ms/sample  (5/90)
[PCA] K=4 seed=42 MLP    F1=0.8659  train=33.6s  inf=0.0015ms/sample  (6/90)
[PCA] K=4 seed=52 DT     F1=0.8893  train=0.2s  inf=0.0001ms/sample  (7/90)
[PCA] K=4 seed=52 RF     F1=0.8891  train=1.9s  inf=0.0027ms/sample  (8/90)
[PCA] K=4 seed=52 XGB    F1=0.8915  train=1.7s  inf=0.0010ms/sample  (9/90)
[PCA] K=4 seed=52 LGBM   F1=0.3388  train=1.1s  inf=0.0011ms/sample  (10/90)
[PCA] K=4 seed=52 KNN    F1=0.7556  train=0.1s  inf=0.0230ms/sample  (11/90)
[PCA] K=4 seed=52 MLP    F1=0.8661  train=16.9s  inf=0.0006ms/sample  (12/90)
[PCA] K=4 seed=62 DT     F1=0.8923

In [5]:
res_df = pd.read_csv(RESULTS_CSV)
print(res_df.groupby(['K', 'classifier'])[['f1', 'train_time_s', 'inference_ms']].agg(['mean', 'std']).to_string())

                     f1           train_time_s           inference_ms          
                   mean       std         mean       std         mean       std
K  classifier                                                                  
4  DT          0.890885  0.001276     0.189009  0.007130     0.000061  0.000004
   KNN         0.753026  0.005528     0.080031  0.001304     0.020704  0.003715
   LGBM        0.781253  0.247353     1.100831  0.079223     0.002673  0.000895
   MLP         0.864536  0.004978    29.489105  9.186463     0.001059  0.000320
   RF          0.890709  0.001226     1.927020  0.120608     0.002733  0.000029
   XGB         0.892325  0.000815     1.711298  0.088673     0.001039  0.000013
8  DT          0.893052  0.000496     0.330079  0.010846     0.000057  0.000001
   KNN         0.750165  0.007323     0.140812  0.002830     0.023152  0.001265
   LGBM        0.892522  0.000771     1.351697  0.056824     0.002802  0.000067
   MLP         0.879794  0.000652    24.